# DINA p' / FF' iteration-by-iteration diagnostics

This notebook loads, visualizes, and quantifies the per-iteration `p_prime.log`
and `ff_prime.log` files produced by the diagnostic patch inserted in
`ptoke1_c` (`src/scenario/neq0_sep_lim_32.f`).

Each row in the two log files is **one equilibrium Picard iteration** — i.e.
one call to `ptoke1_c`, from *any* of the loops that call it, in *either* the
0D breakdown solver (`equil()` / `n_matlab_kav.f`) or the 1D transport-phase
solver (`equil2()` / `n_matlab_kav2.f`). The logging hook sits at the single
common choke point all of these loops pass through, so nothing is missed
regardless of which phase or sub-loop is currently iterating.

**On "does only bootstrap `equil3()` call the solver ~1000 times?"** — no
function named `equil3()` exists in the codebase; the equilibrium kernel
itself is `ptoke1`/`ptoke1_c`, called from many loops inside `equil()` and
`equil2()` with per-loop iteration caps ranging from about 10 to 200. Over a
full run (many timesteps × several sub-stages per timestep) the *total*
number of `ptoke1_c` calls easily reaches the hundreds to low thousands — so
your intuition about the order of magnitude is right, it's just distributed
across many call sites rather than one `equil3()`. Because the log is written
inside `ptoke1_c` itself, every one of those calls — bootstrap-phase or
otherwise — shows up as a row here, and the `ntay` column lets you separate
them by timestep. The grid has `n <= npo = 310` points, comfortably inside the
`400`-wide write format used by the patch, so there is no silent truncation.

Run the cells in order. Only the file paths in the "Load the logs" cell need
to be edited.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path


## 1. Parsing the log files

Format written by the patch (both `p_prime.log` and `ff_prime.log` share it):

```
# DINA p' profile log -- one row per equilibrium Picard iteration (every ptoke1_c call)
# n = <n>
# grid_a <a_1> <a_2> ... <a_n>
# columns: iter ntay tt al1 errm ppx_1 ... ppx_n
<iter> <ntay> <tt> <al1> <errm> <val_1> ... <val_n>
<iter> <ntay> <tt> <al1> <errm> <val_1> ... <val_n>
...
```

`iter` is `i_bound`, DINA's persistent global iteration counter (monotonic
across the whole run). `ntay` is the timestep index the iteration belongs to.
`tt` is physical time, `al1` the current-normalization multiplier, `errm`
DINA's own Picard-convergence residual for that iteration.

In [ ]:
def parse_dina_log(path):
    """Parse a p_prime.log / ff_prime.log file.

    Returns a dict with:
      n          : int, number of radial grid points
      grid_a     : (n,) array, the fixed radial grid (minor-radius-like coordinate)
      meta       : DataFrame with columns iter, ntay, tt, al1, errm
      profile    : (n_iterations, n) array of the logged profile (p' or FF')
    """
    path = Path(path)
    n = None
    grid_a = None
    rows = []

    with open(path, "r") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            if line.startswith("#"):
                if line.startswith("# n ="):
                    n = int(line.split("=")[1].strip())
                elif line.startswith("# grid_a"):
                    grid_a = np.array([float(x) for x in line.split()[2:]])
                # "# columns" and the descriptive header line carry no data
                continue
            rows.append([float(x) for x in line.split()])

    if n is None or grid_a is None:
        raise ValueError(f"{path}: could not find '# n =' / '# grid_a' header lines")

    data = np.array(rows)
    meta = pd.DataFrame(
        {
            "iter": data[:, 0].astype(int),
            "ntay": data[:, 1].astype(int),
            "tt": data[:, 2],
            "al1": data[:, 3],
            "errm": data[:, 4],
        }
    )
    profile = data[:, 5:]
    if profile.shape[1] != n:
        raise ValueError(
            f"{path}: header says n={n} but {profile.shape[1]} profile values were read"
        )

    return {"n": n, "grid_a": grid_a, "meta": meta, "profile": profile}


## 2. Load the logs

Point these at wherever you ran DINA (the working directory of the run, since
that's where `p_prime.log` / `ff_prime.log` get written).

In [ ]:
P_PRIME_LOG  = "/scratch/users/svantnj/p_prime.log"
FF_PRIME_LOG = "/scratch/users/svantnj/ff_prime.log"

pdata  = parse_dina_log(P_PRIME_LOG)
ffdata = parse_dina_log(FF_PRIME_LOG)

print(f"p'  log: {pdata['n']} grid points, {len(pdata['meta'])} iterations, "
      f"ntay range [{pdata['meta'].ntay.min()}, {pdata['meta'].ntay.max()}]")
print(f"FF' log: {ffdata['n']} grid points, {len(ffdata['meta'])} iterations, "
      f"ntay range [{ffdata['meta'].ntay.min()}, {ffdata['meta'].ntay.max()}]")

# quick look at how many ptoke1_c calls happened per timestep -- this is the
# empirical answer to "how many times does the solver actually get called"
calls_per_step = pdata["meta"].groupby("ntay").size()
print("\ncalls per timestep -- min/median/max:",
      calls_per_step.min(), calls_per_step.median(), calls_per_step.max())
print("total calls across the whole run:", len(pdata["meta"]))


## 3. Heatmap: iteration × radial grid

The most information-dense view of "how does the whole profile evolve":
color encodes the profile value (a single sequential scale, one axis of
meaning), x is the radial grid, y is the iteration index. Timestep (`ntay`)
boundaries are drawn as thin horizontal lines so you can see whether a
"weird" jump happens *within* one timestep's Picard loop or *between*
timesteps.

In [ ]:
def profile_heatmap(d, title, colorbar_title):
    fig = go.Figure(
        data=go.Heatmap(
            z=d["profile"],
            x=d["grid_a"],
            y=d["meta"]["iter"],
            colorscale="Viridis",
            colorbar=dict(title=colorbar_title),
            hovertemplate="a=%{x:.4f}<br>iter=%{y}<br>value=%{z:.4e}<extra></extra>",
        )
    )
    # mark timestep boundaries
    step_starts = d["meta"].groupby("ntay")["iter"].min().values
    for it in step_starts[1:]:
        fig.add_hline(y=it, line=dict(color="white", width=0.5, dash="dot"))
    fig.update_layout(
        title=title,
        xaxis_title="grid coordinate a",
        yaxis_title="iteration (i_bound)",
        height=650,
    )
    return fig

profile_heatmap(pdata,  "p' profile vs. Picard iteration",  "p'").show()
profile_heatmap(ffdata, "FF' profile vs. Picard iteration", "FF'").show()


## 4. Animated slider view

Good for scrubbing through individual iterations by hand. With potentially
thousands of iterations, animating *every single frame* is unusable — so this
subsamples to a manageable number of frames (default ~150, evenly spaced in
iteration index) while keeping the heatmap/metrics above lossless.

In [ ]:
def animated_profile(d, title, yaxis_title, n_frames=150):
    meta, profile, grid = d["meta"], d["profile"], d["grid_a"]
    n_total = len(meta)
    stride = max(1, n_total // n_frames)
    idx = np.arange(0, n_total, stride)

    frames = [
        go.Frame(
            data=[go.Scatter(x=grid, y=profile[i], mode="lines")],
            name=str(meta["iter"].iloc[i]),
        )
        for i in idx
    ]

    fig = go.Figure(
        data=[go.Scatter(x=grid, y=profile[idx[0]], mode="lines")],
        frames=frames,
    )
    fig.update_layout(
        title=title,
        xaxis_title="grid coordinate a",
        yaxis_title=yaxis_title,
        height=500,
        updatemenus=[dict(
            type="buttons",
            buttons=[
                dict(label="Play", method="animate",
                     args=[None, {"frame": {"duration": 60, "redraw": True},
                                  "fromcurrent": True}]),
                dict(label="Pause", method="animate",
                     args=[[None], {"frame": {"duration": 0}, "mode": "immediate"}]),
            ],
        )],
        sliders=[dict(
            steps=[
                dict(method="animate",
                     args=[[fr.name], {"frame": {"duration": 0}, "mode": "immediate"}],
                     label=fr.name)
                for fr in frames
            ],
            currentvalue={"prefix": "iter="},
        )],
    )
    return fig

animated_profile(pdata,  "p' profile (scrub through iterations)",  "p'").show()
animated_profile(ffdata, "FF' profile (scrub through iterations)", "FF'").show()


## 5. Strided, color-graded overlay

A "waterfall" style static view: every `stride`-th profile drawn on the same
axes, colored along a single sequential scale keyed to iteration index (early
= dark/cool, late = light/warm, per the chosen colorscale). This is often the
fastest way to *see* whether the profile shape is drifting, oscillating, or
settling, without scrubbing a slider.

In [ ]:
def strided_overlay(d, title, yaxis_title, stride=None, max_lines=60):
    meta, profile, grid = d["meta"], d["profile"], d["grid_a"]
    n_total = len(meta)
    if stride is None:
        stride = max(1, n_total // max_lines)
    idx = np.arange(0, n_total, stride)

    colors = px.colors.sample_colorscale("Viridis", np.linspace(0, 1, len(idx)))

    fig = go.Figure()
    for c, i in zip(colors, idx):
        fig.add_trace(go.Scatter(
            x=grid, y=profile[i], mode="lines",
            line=dict(color=c, width=1.5),
            name=f"iter {meta['iter'].iloc[i]} (ntay {meta['ntay'].iloc[i]})",
            showlegend=False,
            hovertemplate="a=%{x:.4f}<br>value=%{y:.4e}<extra>iter "
                          f"{meta['iter'].iloc[i]}</extra>",
        ))

    # colorbar as a proxy for the legend, since color here is continuous (iteration index)
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers",
        marker=dict(
            colorscale="Viridis", cmin=meta["iter"].iloc[idx[0]],
            cmax=meta["iter"].iloc[idx[-1]], color=[0], showscale=True,
            colorbar=dict(title="iteration"),
        ),
        showlegend=False,
    ))

    fig.update_layout(title=title, xaxis_title="grid coordinate a",
                       yaxis_title=yaxis_title, height=550)
    return fig

strided_overlay(pdata,  "p' profile -- strided overlay",  "p'").show()
strided_overlay(ffdata, "FF' profile -- strided overlay", "FF'").show()


## 6. Scalar diagnostics: `al1` and `errm` vs. iteration

Stacked subplots sharing the x-axis (never a dual-y-axis plot for two
differently-scaled quantities) with timestep boundaries marked, so a spike in
the Picard residual (`errm`) or in the current-normalization multiplier
(`al1`) can be lined up against the heatmap above.

In [ ]:
def scalar_diagnostics(d, title):
    meta = d["meta"]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                         subplot_titles=("al1 (current normalization)",
                                         "errm (Picard residual)"))
    fig.add_trace(go.Scatter(x=meta["iter"], y=meta["al1"], mode="lines",
                              name="al1"), row=1, col=1)
    fig.add_trace(go.Scatter(x=meta["iter"], y=meta["errm"], mode="lines",
                              name="errm"), row=2, col=1)
    fig.update_yaxes(type="log", row=2, col=1, title_text="errm (log scale)")
    fig.update_yaxes(title_text="al1", row=1, col=1)
    fig.update_xaxes(title_text="iteration (i_bound)", row=2, col=1)

    step_starts = meta.groupby("ntay")["iter"].min().values
    for it in step_starts[1:]:
        fig.add_vline(x=it, line=dict(color="gray", width=0.5, dash="dot"),
                       row="all")

    fig.update_layout(title=title, height=600, showlegend=False)
    return fig

scalar_diagnostics(pdata, "p'-log scalar diagnostics").show()


## 7. Shape-change metrics between consecutive iterations

For each consecutive pair of iterations `(k-1, k)` (within the *same*
timestep -- a jump across a `ntay` boundary is a real physical timestep, not
a Picard-convergence artefact, so it's flagged separately rather than folded
into the same metric), compute:

- **L2 (unweighted RMS of the difference)** — `sqrt(mean((f_k - f_{k-1})**2))`.
  Simple, but treats all grid points equally regardless of local grid spacing.
- **L2 (grid-weighted / trapezoidal)** — same idea but integrated with
  `np.trapz` against `grid_a`, so it approximates the continuum
  `∫(Δf)² da` rather than a plain discrete average. More faithful when the
  grid is non-uniform.
- **L∞ (`Lmax`)** — `max(|f_k - f_{k-1}|)`, the worst single grid point. Good
  for catching a localized kink/spike that a bulk L2 norm would wash out.
- **Relative L∞** — `max(|f_k - f_{k-1}|) / max(|f_k|)`, i.e. the same
  construction DINA's own `errm` Picard-convergence criterion uses internally
  (a normalized max-difference). Plotting this alongside DINA's actual
  `errm` column is a good sanity check that the two agree in trend.
- **Shape-similarity (Pearson correlation)** — correlation coefficient
  between `f_{k-1}` and `f_k`. Unlike the norms above, this is insensitive to
  a uniform rescaling of the whole profile and isolates *shape* drift: a
  profile that is simply scaled up/down iteration-to-iteration stays at
  correlation ~1, while a profile that changes curvature/inflection points
  drops noticeably below 1.

In [ ]:
def consecutive_metrics(d):
    meta, profile, grid = d["meta"], d["profile"], d["grid_a"]
    n_iter = profile.shape[0]

    l2 = np.full(n_iter, np.nan)
    l2_weighted = np.full(n_iter, np.nan)
    linf = np.full(n_iter, np.nan)
    rel_linf = np.full(n_iter, np.nan)
    corr = np.full(n_iter, np.nan)
    same_step = np.zeros(n_iter, dtype=bool)

    ntay = meta["ntay"].values
    for k in range(1, n_iter):
        same_step[k] = ntay[k] == ntay[k - 1]
        if not same_step[k]:
            continue  # skip cross-timestep jumps -- not a Picard-convergence event
        prev, cur = profile[k - 1], profile[k]
        diff = cur - prev

        l2[k] = np.sqrt(np.mean(diff ** 2))
        l2_weighted[k] = np.sqrt(np.trapz(diff ** 2, grid) / (grid[-1] - grid[0]))
        linf[k] = np.max(np.abs(diff))
        denom = np.max(np.abs(cur))
        rel_linf[k] = linf[k] / denom if denom > 0 else np.nan
        if np.std(prev) > 0 and np.std(cur) > 0:
            corr[k] = np.corrcoef(prev, cur)[0, 1]

    return pd.DataFrame({
        "iter": meta["iter"].values,
        "ntay": ntay,
        "same_step": same_step,
        "l2": l2,
        "l2_weighted": l2_weighted,
        "linf": linf,
        "rel_linf": rel_linf,
        "corr": corr,
    })

p_metrics  = consecutive_metrics(pdata)
ff_metrics = consecutive_metrics(ffdata)
p_metrics.head()


In [ ]:
def plot_metrics(metrics, title):
    fig = make_subplots(
        rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.05,
        subplot_titles=("L2 (unweighted vs. grid-weighted)", "L-infinity (Lmax)",
                        "Relative L-infinity (compare to DINA's own errm)",
                        "Shape similarity (Pearson correlation, consecutive iters)"),
    )
    fig.add_trace(go.Scatter(x=metrics["iter"], y=metrics["l2"],
                              mode="lines", name="L2 (unweighted)"), row=1, col=1)
    fig.add_trace(go.Scatter(x=metrics["iter"], y=metrics["l2_weighted"],
                              mode="lines", name="L2 (grid-weighted)"), row=1, col=1)
    fig.add_trace(go.Scatter(x=metrics["iter"], y=metrics["linf"],
                              mode="lines", name="Linf"), row=2, col=1)
    fig.add_trace(go.Scatter(x=metrics["iter"], y=metrics["rel_linf"],
                              mode="lines", name="relative Linf"), row=3, col=1)
    fig.add_trace(go.Scatter(x=metrics["iter"], y=metrics["corr"],
                              mode="lines", name="correlation"), row=4, col=1)

    for r in (1, 2, 3):
        fig.update_yaxes(type="log", row=r, col=1)
    fig.update_yaxes(title_text="correlation", range=[0, 1.02], row=4, col=1)
    fig.update_xaxes(title_text="iteration (i_bound)", row=4, col=1)

    step_starts = metrics.loc[metrics["ntay"].diff().fillna(1) != 0, "iter"].values
    for it in step_starts[1:]:
        fig.add_vline(x=it, line=dict(color="gray", width=0.5, dash="dot"), row="all")

    fig.update_layout(title=title, height=950, showlegend=False)
    return fig

plot_metrics(p_metrics,  "p'  -- consecutive-iteration shape-change metrics").show()
plot_metrics(ff_metrics, "FF' -- consecutive-iteration shape-change metrics").show()


## Notes / things to try if the p'/FF' behaviour still looks "weird"

- Cross-reference a spike in `rel_linf` or a dip in `corr` against the `errm`
  panel from Section 6 -- if DINA's own Picard residual is *not* elevated at
  the same iteration, the equilibrium solver thinks it converged even though
  the profile shape moved a lot, which usually points at the boundary
  determination (limiter/X-point switch, `separatrix1`/`second_sep`) rather
  than the p'/FF' closure itself.
- A jump that lines up exactly with a `ntay` boundary (dotted gray lines) is
  a genuine transport timestep update (new p'/FF' from `pp_calc`/`pff_calc`
  after the 1D diffusion solve), not a Picard-iteration artefact -- expected
  to look different from the within-timestep noise.
- If `calls_per_step` (Section 2) varies a lot timestep-to-timestep, that's
  consistent with DINA's per-loop iteration caps (roughly 10-200 depending on
  which sub-stage/loop is active) rather than a fixed count every step.
